In [4]:
import os
import fitz  # PyMuPDF
from tqdm import tqdm
DATA_PATH = "./data/"

def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

documents = []

for file in tqdm(os.listdir(DATA_PATH)):
    if file.endswith(".pdf"):
        full_path = os.path.join(DATA_PATH, file)
        text = extract_text_from_pdf(full_path)
        documents.append({
            "filename": file,
            "text": text
        })

print("Total documents loaded:", len(documents))

100%|████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:04<00:00,  1.50s/it]

Total documents loaded: 3


In [5]:
def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0
    
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start = end - overlap
        
    return chunks

In [6]:
all_chunks = []

for doc in documents:
    chunks = chunk_text(doc["text"])
    for chunk in chunks:
        all_chunks.append({
            "filename": doc["filename"],
            "content": chunk
        })

print("Total chunks created:", len(all_chunks))

Total chunks created: 5831


## model

In [1]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
texts = [chunk["content"] for chunk in all_chunks]
embeddings = model.encode(texts, show_progress_bar=True)
embeddings = np.array(embeddings)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/183 [00:00<?, ?it/s]

Embedding shape: (5831, 384)


In [8]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("Total vectors in index:", index.ntotal)

Total vectors in index: 5831


## connect ollama

In [9]:
import requests
import json
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "mistral"

def generate_answer(context, query):
    
    prompt = f"""
You are a helpful assistant.

Use the following context to answer the question.

Context:
{context}

Question:
{query}

Answer:
"""
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL_NAME,
            "prompt": prompt,
            "stream": False
        }
    )
    
    return response.json()["response"]

## STEP 1 — Generate Alternate Queries

In [13]:
def generate_alternate_queries(user_query, num_queries=3):
    
    prompt = f"""
You are an AI assistant.

Generate {num_queries} different rephrased or related search queries 
that would help retrieve comprehensive information for:

"{user_query}"

Return each query on a new line.
"""

    response = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL_NAME,
            "prompt": prompt,
            "stream": False
        }
    )
    
    output = response.json()["response"]
    
    # Split by line
    queries = [q.strip() for q in output.split("\n") if q.strip()]
    
    return queries

generate_alternate_queries('explain TCP')

['1. "Understand the protocol TCP (Transmission Control Protocol)"',
 '2. "Detailed explanation of TCP in networking"',
 '3. "What is Transmission Control Protocol (TCP) and how does it work?"']

## STEP 2 — Retrieve() For Each Query

In [14]:
def multi_query_retrieve(user_query, top_k=3):
    
    alternate_queries = generate_alternate_queries(user_query)
    
    print("Generated Queries:", alternate_queries)
    
    all_results = []
    
    for query in alternate_queries:
        
        query_embedding = model.encode([query])
        query_embedding = np.array(query_embedding)
        
        distances, indices = index.search(query_embedding, top_k)
        
        for idx in indices[0]:
            all_results.append(all_chunks[idx])
    
    return all_results

## STEP 3 — Remove Duplicate Chunks

In [ ]:
Important.
Because same chunk may appear multiple times.

🧠 Why this works?

Using set for O(1) lookup
Removes repeated context
Prevents token waste

In [15]:
def deduplicate_chunks(chunks):
    
    seen = set()
    unique_chunks = []
    
    for chunk in chunks:
        content = chunk["content"]
        
        if content not in seen:
            unique_chunks.append(chunk)
            seen.add(content)
    
    return unique_chunks

## STEP 4 — Final Multi-Query RAG Pipeline

In [16]:
def multi_query_rag(user_query):
    
    # Step 1: Retrieve
    retrieved_chunks = multi_query_retrieve(user_query)
    
    # Step 2: Deduplicate
    unique_chunks = deduplicate_chunks(retrieved_chunks)
    
    print("Total Unique Chunks:", len(unique_chunks))
    
    # Step 3: Build context
    context = "\n\n".join([chunk["content"] for chunk in unique_chunks])
    
    # Step 4: Final Answer
    answer = generate_answer(context, user_query)
    
    return answer

## test

In [17]:
query = "Explain about optimizers in deep learning"

response = multi_query_rag(query)

print(response)

Generated Queries: ['1. "Deep Learning Optimizers: A Comprehensive Guide"', '2. "Understanding the Role of Optimizers in Deep Neural Networks"', '3. "What are the key types of optimizers used in deep learning and their applications?"']
Total Unique Chunks: 6
 In deep learning, optimizers are algorithms used to minimize the loss function, which measures how well a machine learning model fits the data. Since deep learning models have numerous parameters and complex structures, most of the objective functions do not have analytical solutions. Therefore, we must use numerical optimization algorithms to find the optimal solution.

These optimizers adjust the weights and biases in the neural network iteratively during the training process. By minimizing the loss function, they aim to make the predictions of the model closer to the actual values. Some common optimizers used in deep learning include:

1. Stochastic Gradient Descent (SGD): It is one of the simplest optimization algorithms where